Module 2 - Data Preprocessing & Feature Engineering

Objective

The objective of this phase is to transform the raw Lending Club dataset into a clean and structured dataset suitable for credit risk modelling.

Unlike the exploratory analysis performed previously, this phase focuses on preparing the data for machine learning by removing leakage variables, handling missing values, treating outliers, engineering new features, and building a reusable preprocessing pipeline.

The preprocessing steps will be implemented carefully to avoid data leakage and to ensure the same transformations can later be applied during model deployment.

Preprocessing Roadmap :-

The preprocessing pipeline will be developed in the following order:

1. Load modelling dataset
2. Review current feature set
3. Identify leakage variables
4. Remove identifier columns
5. Handle missing values
6. Handle outliers
7. Feature engineering
8. Feature encoding
9. Feature scaling
10. Train-test split
11. Build reusable preprocessing pipeline

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv(
    "../data/raw/accepted_2007_to_2018Q4.csv",
    low_memory=False
)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [3]:
risk_df = df[
    df["loan_status"].isin(
        ["Fully Paid", "Charged Off"]
    )
].copy()

risk_df["target"] = np.where(
    risk_df["loan_status"] == "Charged Off",
    1,
    0
)

print(risk_df.shape)

risk_df["target"].value_counts()

(1345310, 152)


target
0    1076751
1     268559
Name: count, dtype: int64

Feature Audit

Before preprocessing begins, all available variables are reviewed to identify:

- Identifier variables
- Target leakage variables
- High missing value variables
- Business relevant features
- Features requiring transformation

Only after this review will features be removed from the modelling dataset.

In [4]:
feature_audit = pd.DataFrame({
    "Feature": risk_df.columns,
    "Data Type": risk_df.dtypes.values,
    "Missing Values": risk_df.isnull().sum().values,
    "Missing (%)": (
        risk_df.isnull().mean()*100
    ).values
})

feature_audit.sort_values(
    by="Missing (%)",
    ascending=False
).head(50)

,Feature,Data Type,Missing Values,Missing (%)
1,member_id,float64,1345310,100.000000
49,next_pymnt_d,str,1345310,100.000000
140,orig_projected_additional_accrued_interest,float64,1341551,99.720585
133,hardship_amount,float64,1339556,99.572292
142,hardship_last_payment_amount,float64,1339556,99.572292
137,hardship_length,float64,1339556,99.572292
131,hardship_status,str,1339556,99.572292
130,hardship_reason,str,1339556,99.572292
132,deferral_term,float64,1339556,99.572292
134,hardship_start_date,str,1339556,99.572292


In [5]:
feature_audit.to_csv(
    "../reports/feature_audit.csv",
    index=False
)

print("Feature audit exported successfully.")

Feature audit exported successfully.


Feature Categorization

Before removing any variables, the features are categorized based on their business relevance and their suitability for predictive modelling.

The categories include:

- Identifier Features
- Target Leakage Features
- Joint Application Features
- High Missing Features
- Features to Retain
- Features Requiring Further Investigation

This approach ensures that feature removal decisions are based on both business understanding and data quality rather than only missing value percentages.

In [6]:
identifier_columns = [
    "id",
    "member_id",
    "url"
]

identifier_columns

['id', 'member_id', 'url']

In [7]:
leakage_columns = [

    "next_pymnt_d",

    "hardship_type",
    "hardship_reason",
    "hardship_status",
    "deferral_term",
    "hardship_amount",
    "hardship_start_date",
    "hardship_end_date",
    "payment_plan_start_date",
    "hardship_length",
    "hardship_dpd",
    "hardship_loan_status",
    "hardship_payoff_balance_amount",
    "hardship_last_payment_amount",
    "orig_projected_additional_accrued_interest",

    "debt_settlement_flag_date",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    "settlement_term"
]

len(leakage_columns)

21

In [8]:
joint_application_columns = [

    "annual_inc_joint",
    "verification_status_joint",
    "dti_joint",

    "revol_bal_joint",

    "sec_app_fico_range_low",
    "sec_app_fico_range_high",
    "sec_app_earliest_cr_line",

    "sec_app_inq_last_6mths",
    "sec_app_mort_acc",
    "sec_app_open_acc",
    "sec_app_revol_util",
    "sec_app_open_act_il",
    "sec_app_num_rev_accts",
    "sec_app_chargeoff_within_12_mths",
    "sec_app_collections_12_mths_ex_med",
    "sec_app_mths_since_last_major_derog"

]

len(joint_application_columns)

16

In [9]:
print("Identifier Columns :", len(identifier_columns))
print("Leakage Columns :", len(leakage_columns))
print("Joint Application Columns :", len(joint_application_columns))

Identifier Columns : 3
Leakage Columns : 21
Joint Application Columns : 16


Initial Feature Removal Strategy

Based on the feature audit, three groups of variables have been identified for removal during preprocessing.

1. Identifier variables (`id`, `member_id`, `url`) do not contain predictive information and are only used for record identification.

2. Hardship, settlement and future payment related variables contain information that becomes available only after loan issuance. Including these variables would introduce target leakage and artificially inflate model performance.

3. Joint application variables have more than 98% missing values because the majority of loans are single-applicant loans. Retaining these variables would add unnecessary complexity while contributing little predictive value.

In [10]:
all_columns_to_remove = (
    identifier_columns
    + leakage_columns
    + joint_application_columns
)

missing_columns = [
    col for col in all_columns_to_remove
    if col not in risk_df.columns
]

print("Total Columns Planned for Removal :", len(all_columns_to_remove))
print("Columns Not Found :", len(missing_columns))

if missing_columns:
    print(missing_columns)
else:
    print("All columns exist.")

Total Columns Planned for Removal : 40
Columns Not Found : 0
All columns exist.


Dataset Before Feature Removal

The initial modelling dataset is recorded before removing non-predictive and leakage variables.
Recording the dataset dimensions before each major preprocessing step helps track the impact of feature reduction throughout the project.

In [11]:
print("Rows :", risk_df.shape[0])
print("Columns :", risk_df.shape[1])

Rows : 1345310
Columns : 152


In [12]:
risk_df = risk_df.drop(
    columns=all_columns_to_remove
)

print("Columns removed successfully.")
print("Rows :", risk_df.shape[0])
print("Columns :", risk_df.shape[1])

Columns removed successfully.
Rows : 1345310
Columns : 112


In [13]:
remaining = [
    col for col in all_columns_to_remove
    if col in risk_df.columns
]

print(remaining)

[]


 Saving Intermediate Dataset

After removing identifier, leakage and joint application features, the cleaned dataset is saved as an intermediate version. This ensures that subsequent preprocessing steps operate on a consistent dataset without modifying the original raw data.

In [14]:
risk_df.to_csv(
    "../data/processed/credit_risk_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


In [15]:
import os

os.path.exists("../data/processed/credit_risk_cleaned.csv")

True